In [ ]:
import duckdb
import numpy as np
import matplotlib.pyplot as plt

# One or more .parquet files in this chunk
path = "/home/bien/Documents/Development/RCS/datasets/data_lerobot/utn_usbc_insertion_nonbinary/data/chunk-000/*.parquet"

con = duckdb.connect()

# Inspect types and available episodes
display(con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{path}')
""").df())

episodes = con.execute(f"""
    SELECT
        episode_index,
        COUNT(*) AS n_frames,
        MIN(timestamp) AS start_timestamp,
        MAX(timestamp) AS end_timestamp
    FROM read_parquet('{path}')
    GROUP BY episode_index
    ORDER BY episode_index
""").df()

display(episodes)

In [ ]:
episode_id = 0  # Change this to an episode_index shown above

df = con.execute(f"""
    SELECT
        timestamp,
        frame_index,
        episode_index,
        "observation.state" AS observation_state,
        action
    FROM read_parquet('{path}')
    WHERE episode_index = ?
    ORDER BY timestamp, frame_index
""", [episode_id]).df()

# Convert each vector-valued Parquet cell into a NumPy matrix:
# shape: (number_of_frames, vector_dimension)
state = np.stack(df["observation_state"].to_numpy())
action = np.stack(df["action"].to_numpy())

# Relative seconds are easier to read than absolute timestamps
time_s = df["timestamp"].to_numpy() - df["timestamp"].iloc[0]

print(f"Episode {episode_id}: {len(df)} frames")
print("state shape :", state.shape)
print("action shape:", action.shape)

In [ ]:
joint_labels = {
    0: "joint_1",
    1: "joint_2",
    2: "joint_3",
    3: "joint_4",
    4: "joint_5",
    5: "joint_6",
    6: "joint_7",
    7: "gripper",
}

fig, axes = plt.subplots(
    2, 1,
    figsize=(16, 10),
    sharex=True,
    constrained_layout=True
)

for dim in range(state.shape[1]):
    label = f"obs: {joint_labels.get(dim, f'dim_{dim}')}"
    axes[0].plot(time_s, state[:, dim], linewidth=1, label=label)

axes[0].set_title(f"Observation state — episode {episode_id}")
axes[0].set_ylabel("Position")
axes[0].grid(alpha=0.3)
axes[0].legend(ncol=4, fontsize=9)

for dim in range(action.shape[1]):
    label = f"action: {joint_labels.get(dim, f'dim_{dim}')}"
    axes[1].plot(time_s, action[:, dim], linewidth=1, label=label)

axes[1].set_title(f"Actions — episode {episode_id}")
axes[1].set_xlabel("Time since episode start (s)")
axes[1].set_ylabel("Command")
axes[1].grid(alpha=0.3)
axes[1].legend(ncol=4, fontsize=9)

plt.show()